# Gaussian Mixture Models

O **GMM** (*Gaussian Mixture Model*) supõe que os dados foram gerados por uma mistura de $K$ distribuições gaussianas, cada uma com sua própria média, covariância e peso. Em vez de atribuir cada ponto a um único cluster, ele calcula a probabilidade de cada ponto ter vindo de cada componente. É a versão probabilística do K-Means: os clusters podem ser alongados e inclinados, e as fronteiras entre eles deixam de ser rígidas.

## Fundamentação Matemática

Um GMM modela a densidade dos dados como uma soma ponderada de $K$ gaussianas:

$$p(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$$

- $\pi_k$: peso do componente $k$, a fração dos dados que ele explica, com $\sum_k \pi_k = 1$
- $\boldsymbol{\mu}_k$: média (centro) do componente $k$
- $\boldsymbol{\Sigma}_k$: matriz de covariância, que define o formato e a orientação do componente $k$

Cada componente é uma gaussiana multivariada em $D$ dimensões:

$$\mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}, \boldsymbol{\Sigma}) = \frac{1}{(2\pi)^{D/2} \, |\boldsymbol{\Sigma}|^{1/2}} \exp\left( -\frac{1}{2} (\mathbf{x} - \boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu}) \right)$$

Queremos os parâmetros $\{\pi_k, \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k\}$ que melhor explicam os dados. Se soubéssemos de qual componente veio cada ponto, bastaria calcular a média e a covariância de cada grupo. Como não sabemos, estimamos isso de forma iterativa.

### O Algoritmo EM

O **Expectation-Maximization** alterna dois passos, assim como o K-Means, mas com atribuições probabilísticas.

**Passo E (Expectation)**: quanto cada gaussiana explica cada ponto? A **responsabilidade** $\gamma_{ik}$ é a probabilidade de o ponto $\mathbf{x}_i$ ter vindo do componente $k$, dados os parâmetros atuais:

$$\gamma_{ik} = \frac{\pi_k \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)}{\sum_{j=1}^{K} \pi_j \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_j, \boldsymbol{\Sigma}_j)}$$

**Passo M (Maximization)**: recalcula os parâmetros usando as responsabilidades como pesos. Com $N_k = \sum_i \gamma_{ik}$, o número efetivo de pontos do componente $k$:

$$\pi_k = \frac{N_k}{N} \qquad \boldsymbol{\mu}_k = \frac{1}{N_k} \sum_{i=1}^{N} \gamma_{ik} \, \mathbf{x}_i \qquad \boldsymbol{\Sigma}_k = \frac{1}{N_k} \sum_{i=1}^{N} \gamma_{ik} \, (\mathbf{x}_i - \boldsymbol{\mu}_k)(\mathbf{x}_i - \boldsymbol{\mu}_k)^\top$$

Repete-se até os parâmetros pararem de mudar. Se forçássemos $\gamma_{ik} \in \{0, 1\}$ e fixássemos $\boldsymbol{\Sigma}_k = \sigma^2 I$, o EM se reduziria exatamente ao K-Means.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.stats import multivariate_normal, mode
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## Implementação

A classe segue as equações acima. A densidade $\mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}, \boldsymbol{\Sigma})$ vem pronta do SciPy (`multivariate_normal.pdf`); o resto é o EM: inicializa as médias com pontos aleatórios do conjunto, as covariâncias com a identidade e os pesos uniformes, e então alterna os passos E e M por um número fixo de iterações.

In [ ]:
class GMM:
    def __init__(self, n_components=3, n_iter=100, random_state=42):
        self.n_components = n_components
        self.n_iter = n_iter
        self.random_state = random_state

    def initialize(self, X):
        N, D = X.shape
        K = self.n_components
        rng = np.random.default_rng(self.random_state)

        self.weights = np.full(K, 1 / K)                     # (K,)      um peso por componente
        self.means = X[rng.choice(N, K, replace=False)]      # (K, D)    um centro por componente
        self.covariances = np.array([np.eye(D)] * K)         # (K, D, D) uma matriz por componente

    def e_step(self, X):
        # Quanto cada gaussiana explica cada ponto?
        resp = np.zeros((len(X), self.n_components))         # (N, K)

        for k in range(self.n_components):
            resp[:, k] = self.weights[k] * multivariate_normal.pdf(X, self.means[k], self.covariances[k])

        return resp / resp.sum(axis=1, keepdims=True)

    def m_step(self, X, resp):
        # Recalcula os parâmetros usando as responsabilidades como pesos.
        N_k = resp.sum(axis=0)                               # (K,) número efetivo de pontos por componente

        self.weights = N_k / len(X)
        self.means = (resp.T @ X) / N_k[:, np.newaxis]

        for k in range(self.n_components):
            diff = X - self.means[k]
            self.covariances[k] = (resp[:, k] * diff.T) @ diff / N_k[k]

    def fit(self, X):
        self.initialize(X)

        for _ in range(self.n_iter):
            resp = self.e_step(X)
            self.m_step(X, resp)

        self.resp = self.e_step(X)
        self.labels = self.resp.argmax(axis=1)
        return self

    def predict(self, X):
        return self.e_step(X).argmax(axis=1)

### Dados Sintéticos

Três grupos gaussianos com formatos e tamanhos diferentes: um grande, alongado e inclinado, um vertical e um pequeno e esférico. É o cenário em que a covariância importa.

Como vamos desenhar o modelo várias vezes ao longo do notebook, isolamos a plotagem numa função, que pinta cada ponto pela componente mais provável e desenha as elipses de 1, 2 e 3 desvios-padrão de cada gaussiana.

In [ ]:
rng = np.random.default_rng(42)

X_synthetic = np.vstack([
    rng.multivariate_normal([0, 0], [[5.0, 4.0], [4.0, 3.5]], 300),      # alongado e inclinado
    rng.multivariate_normal([3.5, -2.5], [[0.6, 0.0], [0.0, 4.0]], 200),  # vertical
    rng.multivariate_normal([-3, 4], [[0.5, 0.0], [0.0, 0.5]], 100),      # pequeno e esférico
])
true_labels = np.repeat([0, 1, 2], [300, 200, 100])

print(f"Shape: {X_synthetic.shape}")

In [ ]:
def draw_ellipses(mean, cov, ax, color):
    # Elipses de 1, 2 e 3 desvios-padrão: os eixos são os autovetores da covariância.
    eigvals, eigvecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
    width, height = 2 * np.sqrt(eigvals)

    for n_std in [1, 2, 3]:
        ax.add_patch(Ellipse(mean, n_std * width, n_std * height, angle=angle,
                             facecolor='none', edgecolor=color, linewidth=1.5, alpha=0.6))


def plot_gmm(X, gmm, ax, title):
    palette = plt.cm.tab10.colors
    labels = gmm.predict(X)

    for k in range(gmm.n_components):
        mask = labels == k
        ax.scatter(X[mask, 0], X[mask, 1], color=palette[k], s=30, alpha=0.6, label=f'Componente {k}')
        draw_ellipses(gmm.means[k], gmm.covariances[k], ax, palette[k])

    ax.scatter(gmm.means[:, 0], gmm.means[:, 1], c='black', marker='x', s=100, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(X_synthetic[:, 0], X_synthetic[:, 1], c=true_labels, cmap='tab10', vmax=9, s=30, alpha=0.6)
axes[0].set_title('Estrutura Real')

axes[1].scatter(X_synthetic[:, 0], X_synthetic[:, 1], c='black', s=30, alpha=0.6)
axes[1].set_title('Dados sem Rótulos')

for ax in axes:
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

### Primeira Execução

In [ ]:
gmm = GMM(n_components=3).fit(X_synthetic)

fig, ax = plt.subplots(figsize=(7, 6))
plot_gmm(X_synthetic, gmm, ax, 'GMM (K=3)')
ax.legend()
plt.show()

print("Pesos:", gmm.weights.round(3))
print("Médias:\n", gmm.means.round(2))

O modelo recupera os três grupos. Os pesos 0.50, 0.17 e 0.33 são as proporções reais, 300, 100 e 200 pontos em 600, e as médias caem a menos de 0.1 dos centros verdadeiros. As elipses mostram o que o K-Means não tem: cada componente aprende o próprio formato, e o grupo inclinado é descrito por uma covariância com forte correlação entre as duas features.

### Visualizando o EM

Como a inicialização é determinística, rodar o modelo com `n_iter=n` reproduz exatamente o estado após $n$ iterações, o que permite ver as elipses se ajustando.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, n in zip(axes, [1, 5, 10, 20]):
    partial = GMM(n_components=3, n_iter=n).fit(X_synthetic)
    plot_gmm(X_synthetic, partial, ax, f'Iteração {n}')

plt.tight_layout()
plt.show()

Na primeira iteração as três gaussianas são quase iguais: partiram da identidade e, após um único passo M, cada uma cobre boa parte dos dados. Dali em diante os componentes se especializam. O vertical, que é o mais denso, é o primeiro a se destacar, e o alongado só assume o formato final quando o pequeno se desprende dele, por volta da iteração 10. Na iteração 20 o modelo já está no estado final: as 80 iterações restantes não mudam nada.

### Atribuições Suaves

O que distingue o GMM é que a saída não é um rótulo, mas uma distribuição de probabilidade sobre os componentes. A matriz de responsabilidades tem uma linha por ponto e uma coluna por componente, e cada linha soma 1.

In [ ]:
resp = gmm.resp
confidence = resp.max(axis=1)

print("Responsabilidades dos 5 primeiros pontos:")
print(resp[:5].round(3))
print(f"\nPontos com confiança abaixo de 90%: {np.sum(confidence < 0.9)} de {len(X_synthetic)}")

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_synthetic[:, 0], X_synthetic[:, 1], c=confidence, cmap='viridis', s=30, alpha=0.8)
plt.colorbar(sc, label='Probabilidade do componente mais provável')
ax.set_title('Confiança da Atribuição')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
plt.show()

Os primeiros pontos pertencem ao componente 0 com probabilidade praticamente 1, mas o terceiro já divide 4% com o componente 2. Só 10 dos 600 pontos têm confiança abaixo de 90%, e todos estão na região em que a ponta do grupo alongado encosta no vertical.

É a diferença entre uma fronteira dura e uma suave. Longe da fronteira, o GMM se comporta como o K-Means; perto dela, ele diz que não sabe, em vez de fingir certeza. Essa incerteza é informação útil: dá para descartar os pontos ambíguos, ponderar decisões por ela ou usá-la para detectar amostras atípicas.

## Comparação com o K-Means

O K-Means é um caso particular do GMM: assume que todo cluster é uma esfera do mesmo tamanho. Quando os grupos são alongados ou têm tamanhos diferentes, essa suposição custa caro. Para ver a diferença, pintamos o fundo com a região de decisão de cada modelo.

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_synthetic)

x_min, x_max = X_synthetic[:, 0].min() - 1, X_synthetic[:, 0].max() + 1
y_min, y_max = X_synthetic[:, 1].min() - 1, X_synthetic[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].contourf(xx, yy, kmeans.predict(grid).reshape(xx.shape), alpha=0.2, cmap='tab10', vmax=9)
axes[0].scatter(X_synthetic[:, 0], X_synthetic[:, 1], c=kmeans.labels_, cmap='tab10', vmax=9, s=30, alpha=0.6)
axes[0].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], c='black', marker='x', s=100, linewidth=2)
axes[0].set_title('K-Means (K=3)')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

axes[1].contourf(xx, yy, gmm.predict(grid).reshape(xx.shape), alpha=0.2, cmap='tab10', vmax=9)
plot_gmm(X_synthetic, gmm, axes[1], 'GMM (K=3)')

plt.tight_layout()
plt.show()

O K-Means corta a ponta do grupo alongado e a entrega ao vertical: 76 dos 300 pontos mudam de lado, e a pureza cai para 87%, contra 99% do GMM. As fronteiras explicam o resultado. As do K-Means são retas, equidistantes dos centróides. As do GMM são curvas, porque comparam distâncias de Mahalanobis: um ponto na ponta do grupo alongado está longe da média em distância euclidiana, mas perto quando medido na escala da própria elipse.

## Aplicação ao Iris

Voltamos ao Iris, desta vez com o comprimento da sépala e a largura da pétala. Nesse par de features, *versicolor* e *virginica* formam nuvens alongadas, inclinadas e encostadas uma na outra, exatamente o formato que o K-Means não consegue modelar.

In [ ]:
iris = load_iris()
X_iris = iris.data[:, [0, 3]]
y_iris = iris.target


def purity(labels, y_true):
    # Cada cluster recebe a classe mais frequente dentro dele.
    hits = sum(np.sum(y_true[labels == k] == mode(y_true[labels == k], keepdims=True)[0][0])
               for k in np.unique(labels))
    return hits / len(y_true)


gmm_iris = GMM(n_components=3).fit(X_iris)
kmeans_iris = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_iris)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

axes[0].scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, cmap='tab10', vmax=9, s=40, alpha=0.7)
axes[0].set_title('Espécies Reais')

axes[1].scatter(X_iris[:, 0], X_iris[:, 1], c=kmeans_iris.labels_, cmap='tab10', vmax=9, s=40, alpha=0.7)
axes[1].set_title(f'K-Means: pureza {purity(kmeans_iris.labels_, y_iris):.1%}')

plot_gmm(X_iris, gmm_iris, axes[2], f'GMM: pureza {purity(gmm_iris.labels, y_iris):.1%}')

for ax in axes:
    ax.set_xlabel('Comprimento da Sépala (cm)')
    ax.set_ylabel('Largura da Pétala (cm)')

plt.tight_layout()
plt.show()

O GMM chega a 96% de pureza, contra 83% do K-Means. Com um centróide por grupo, o K-Means separa *versicolor* de *virginica* por uma reta quase vertical e erra os pontos das duas espécies que a cruzam. O GMM aprende que as duas nuvens são elipses inclinadas na mesma direção e as separa ao longo da diagonal. Os erros que restam estão na região em que as espécies de fato se sobrepõem, onde nenhum modelo baseado só nessas duas medidas consegue decidir.

## Custo e Limitações

Cada iteração do EM custa $O(NKD^2)$, dominado pela avaliação das $K$ gaussianas em todos os pontos. Como o número de parâmetros de covariância cresce com $D^2$, em dimensão alta o modelo fica caro e instável. O Scikit-Learn oferece o `covariance_type` (`'diag'`, `'spherical'`, `'tied'`) para restringir o formato das gaussianas e reduzir esse custo.

Nossa implementação roda um número fixo de iterações. Na prática mede-se a **log-verossimilhança** $\sum_i \log p(\mathbf{x}_i)$, que nunca diminui de uma iteração para a outra, e o algoritmo para quando ela deixa de crescer. A mesma quantidade serve para escolher $K$: como ela sempre melhora com mais componentes, usa-se o **BIC**, que a penaliza pelo número de parâmetros (`GaussianMixture.bic()` no Scikit-Learn).

Como o K-Means, o EM só encontra máximos locais e depende da inicialização. O Scikit-Learn inicializa as médias com o K-Means e roda o algoritmo várias vezes (`n_init`), ficando com a melhor log-verossimilhança.

Há um problema exclusivo do GMM: um componente pode colapsar sobre um único ponto, com a covariância tendendo a zero e a densidade a infinito. É uma **singularidade** da verossimilhança. O Scikit-Learn a evita somando uma constante pequena à diagonal da covariância (`reg_covar`).

Por fim, o modelo assume que cada grupo é uma gaussiana. Formas como as luas ou os anéis não são bem descritas por uma única elipse, e o GMM se comporta de maneira tão ruim quanto o K-Means nelas. Nesses casos, ele ainda é útil como **estimador de densidade**: com componentes suficientes, a mistura aproxima qualquer distribuição contínua, mesmo sem que cada componente corresponda a um cluster.

## Exercícios

### Exercício 1: Wine Dataset

Carregue o dataset `wine` do Scikit-Learn, padronize as 13 features com o `StandardScaler` e treine o GMM com $K=3$. Calcule a pureza em relação aos cultivares e compare com o K-Means. Quantos parâmetros tem cada covariância completa em 13 dimensões, e por que isso é um risco num conjunto de apenas 178 amostras?

In [ ]:
# Seu código aqui

### Exercício 2: Covariância Diagonal

Crie uma subclasse de `GMM` que sobrescreva o `m_step` para manter apenas a diagonal de cada covariância (`np.diag(np.diag(cov))`), que é o `covariance_type='diag'` do Scikit-Learn. Aplique aos dados sintéticos deste notebook e compare com o modelo de covariância completa, visualmente e pela pureza. Por que o componente inclinado é o mais afetado, e quantos parâmetros a versão diagonal economiza?

In [ ]:
# Seu código aqui

### Exercício 3: Detecção de Anomalias

Um GMM treinado é um estimador de densidade: a fórmula $p(\mathbf{x}) = \sum_k \pi_k \, \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$ pode ser calculada com `multivariate_normal.pdf` e os atributos `weights`, `means` e `covariances` do modelo. Pontos com densidade muito baixa são candidatos a anomalias.

Nos dados abaixo, três grupos normais mais 30 pontos espalhados uniformemente, treine um GMM com $K=3$, calcule a densidade de cada ponto e marque como anomalia os que ficarem abaixo de um limiar (comece pelo percentil 5). Plote os dados destacando as anomalias detectadas e compare com as anomalias reais. O que muda ao variar o limiar?

In [ ]:
rng = np.random.default_rng(7)

X_normal = np.vstack([rng.multivariate_normal([0, 0], [[1.0, 0.5], [0.5, 1.0]], 200),
                      rng.multivariate_normal([7, 7], [[1.5, 0.0], [0.0, 0.5]], 150),
                      rng.multivariate_normal([-6, 6], [[0.8, 0.0], [0.0, 0.8]], 100)])
X_outliers = rng.uniform([-12, -6], [14, 14], (30, 2))

X_anomaly = np.vstack([X_normal, X_outliers])
y_anomaly = np.r_[np.zeros(len(X_normal)), np.ones(len(X_outliers))]   # 1 = anomalia

plt.figure(figsize=(7, 6))
plt.scatter(X_anomaly[:, 0], X_anomaly[:, 1], c='black', s=30, alpha=0.6)
plt.title('Dados com Anomalias')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

In [ ]:
# Seu código aqui